In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

# Question 7

In `./data/ames_prices.csv`, create an indicator that equals 1 when `Central.Air == 'Y'` and 0 otherwise.

In [ ]:
df = pd.read_csv('data/ames_prices.csv')
df['Central.Air'].value_counts(dropna=False)

Central.Air
Y    2734
N     196
Name: count, dtype: int64

In [11]:
central_air = (df['Central.Air'] == 'Y').astype(int).to_numpy()

### a. Estimate the proportion of homes with central air.

In [12]:
p_hat = central_air.mean()
print(p_hat)

0.9331058020477816


93.31% of homes have central air

### b. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.

In [ ]:
seed = 54321
B = 2000
rng = np.random.default_rng(seed)

boot_est = np.array([rng.choice(central_air, size=len(central_air), replace=True).mean() for i in range(B)])

`seed = 54321`, `B = 2000`

### c. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

In [15]:
boot_se = boot_est.std(ddof=1)
print(boot_se)

0.004582862060179471


### d. Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals.

In [18]:
for interval in [0.90, 0.95, 0.99]:
    z = stats.norm.ppf(1 - (1 - interval) / 2)
    low, high = p_hat - z * boot_se, p_hat + z * boot_se
    print(f"{interval:.0%}: z={z:.3f}, [{low:.4f}, {high:.4f}]")

90%: z=1.645, [0.9256, 0.9406]
95%: z=1.960, [0.9241, 0.9421]
99%: z=2.576, [0.9213, 0.9449]


### e. Use each confidence interval to evaluate $H_0:p=.90$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.

None of the confidence intervals include p = 0.90, so we reject the null in all cases. This is strong evidence that the percentage of homes with central air is above 90%, matching the estimate of 93.31% calculated in part a.

# Question 8

In `./data/ames_prices.csv`, estimate the Pearson correlation between `np.arcsinh(area)` and `np.arcsinh(price)`.

### a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the correlation by resampling paired rows. Report the seed and number of bootstrap replications.

In [ ]:
df = pd.read_csv('./data/ames_prices.csv')

a = np.arcsinh(df['area']).dropna()
y = np.arcsinh(df['price']).dropna()
n = len(a)

r_hat = np.corrcoef(a, y)[0, 1]
print(r_hat)

0.7231661530091358


In [29]:
seed = 12345
B = 2000
np.random.seed(seed)

boot_est = np.zeros(B)
for b in range(B):
    i = rng.integers(0, n, size=n)
    boot_est[b] = np.corrcoef(a[i], y[i])[0, 1]

`seed = 12345`, `B = 2000`

### b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the $1/(N−1)$ degrees of freedom correction).

In [30]:
boot_se = np.std(boot_est, ddof=1)
print(boot_se)

0.010055948670914987


### c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [31]:
for interval in [0.90, 0.95, 0.99]:
    z = stats.norm.ppf(1 - (1 - interval) / 2)
    low, high = r_hat - z * boot_se, r_hat + z * boot_se
    print(f"{interval:.0%}: z={z:.3f}, [{low:.4f}, {high:.4f}]")

90%: z=1.645, [0.7066, 0.7397]
95%: z=1.960, [0.7035, 0.7429]
99%: z=2.576, [0.6973, 0.7491]


### d. Use each confidence interval to evaluate $H_0: \rho = 0$. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret the association without making a causal claim.

None of the confidence intervals include p = 0, so we reject the null in all cases. This is strong evidence that larger homes sell for a higher price. However, this may not only be due to the size of the house; larger houses often have more bedrooms and bathrooms, modern appliances, and other features that make them more expensive. 

### e. Briefly explain why `area` and `price` must remain paired during resampling.

`area` and `price` are paired together with the same house, so resampling must take the values from the same observation, as they are dependant on each other. Resampling them separaely would not allow us to form any conclusions on their relation to each other.

# Question 9

In `./data/metabric.csv`, use `Overall Survival (Months)` to estimate the mean recorded time to death or last follow-up. Include both living and deceased patients.

### a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

In [32]:
df = pd.read_csv('./data/metabric.csv')

df['Overall Survival (Months)'].isna().sum()
df['Overall Survival Status'].value_counts()

Overall Survival Status
1:DECEASED    768
0:LIVING      575
Name: count, dtype: int64

In [39]:
x = df['Overall Survival (Months)']
n = len(x)
mu_hat = x.mean()

print(mu_hat)

129.09429138744105


In [40]:
seed = 67890
B = 2000
rng = np.random.default_rng(seed)

boot_est = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(B)])

`seed = 67890`, `B = 2000`

### b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the $1/(N−1)$ degrees of freedom correction).

In [41]:
boot_se = boot_est.std(ddof=1)
print(boot_se)

2.1099968014387342


### c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [43]:
for interval in [0.90, 0.95, 0.99]:
    z = stats.norm.ppf(1 - (1 - interval) / 2)
    low, high = mu_hat - z * boot_se, mu_hat + z * boot_se
    print(f"{interval:.0%}: z={z:.3f}, [{low:.4f}, {high:.4f}]")

90%: z=1.645, [125.6237, 132.5649]
95%: z=1.960, [124.9588, 133.2298]
99%: z=2.576, [123.6593, 134.5293]


### d. Use each confidence interval to evaluate $H_0: \mu = 120$ months. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis.

The null hypothesis $\mu = 120$ months is not in any of the confidence intervals, so we reject the null in all cases.

### e. Interpret the estimate and confidence intervals in both months and years. Because living patients' eventual death times are right-censored, explain why these results do not estimate unrestricted mean lifetime.

The average recorded time between diagnosis and death or last follow-up was approximately 129.1 months (10.8 years), with a 95% confidence interval of 125.0–133.2 months (10.4–11.1 years). The null hypothesis that the population mean is 120 months (10 years) is rejected at all three significance levels, providing evidence that the mean recorded time exceeds 10 years. However, these results do not represent unrestricted mean lifetime because patients who are still alive have right-censored survival times. Their recorded times reflect the duration from diagnosis to their last follow-up, rather than their eventual time of death, so their actual lifetimes may be longer than the observed values.

# Question 10

In `./data/nhanes_data_17_18.csv`, select respondents who are at least 18 years old and have an observed value for `SleepHoursWeekdaysOrWorkdays`. Estimate their mean weekday sleep.

### a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

In [44]:
df = pd.read_csv('data/nhanes_data_17_18.csv')

d = df[(df['AgeInYearsAtScreening'] >= 18) & df['SleepHoursWeekdaysOrWorkdays'].notna()]

/var/folders/7k/_zt4krl53h95mn99bw4jzgnc0000gn/T/ipykernel_25042/3023647096.py:1: DtypeWarning: Columns (0: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/nhanes_data_17_18.csv')


In [45]:
x = d['SleepHoursWeekdaysOrWorkdays'].to_numpy()
n = len(x)
mu_hat = x.mean()
print(mu_hat)

7.62124248496994


In [47]:
seed = 9876
B = 2000
rng = np.random.default_rng(seed)

boot_est = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(B)])

`seed = 9876`, `B = 2000`

### b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (`ddof=1`).

In [48]:
boot_se = boot_est.std(ddof=1)
print(boot_se)

0.022617319697799988


### c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [49]:
for interval in [0.90, 0.95, 0.99]:
    z = stats.norm.ppf(1 - (1 - interval) / 2)
    low, high = mu_hat - z * boot_se, mu_hat + z * boot_se
    print(f"{interval:.0%}: z={z:.3f}, [{low:.4f}, {high:.4f}]")

90%: z=1.645, [7.5840, 7.6584]
95%: z=1.960, [7.5769, 7.6656]
99%: z=2.576, [7.5630, 7.6795]


### d. Use each confidence interval to evaluate $H_0: \mu = 8$ hours. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis.

The null hypothesis $\mu = 8$ hours is not in any of the confidence intervals, so we reject the null in all cases.

### e. Interpret the result and discuss whether its statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted. Because the complex survey design is ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

The average weekday sleep duration among adults in this sample was 7.62 hours. The null hypothesis is rejected at all three significance levels, indicating strong statistical evidence that the mean differs from 8 hours. The small standard error of 0.023 hours suggests that the estimated mean is quite precise. However, the confidence intervals place the average only about 20–25 minutes below the recommended 8 hours of sleep. Thus, although the difference is statistically significant, its practical importance may be limited. These conclusions apply to the sample analyzed and should not be interpreted as nationally representative because the complex survey design was not accounted for.